RUNTIME: H100 recommended, A100 fallback, T4 only if necessary
INSTRUCTIONS:

1. Set runtime to GPU: Runtime → Change runtime type → GPU → H100 (or A100 if unavailable)
2. Run this notebook from inside the cloned repo (e.g., `/content/<repo>/notebooks`) so `src/` imports work
3. Ensure repo `data/` contains processed splits/fractions from `00_eda` before training
4. Use the resumable runner cells (`run_and_persist_one`, `run_pending`) instead of one huge sweep
5. Persist outputs incrementally to `results/` and resume from pending runs after crashes
6. Save notebook state regularly (Ctrl+S); optional full-data checkpoints can be written to Drive


In [1]:
# ── 1. Install dependencies ───────────────────────────────────────
# Pin compatible versions to avoid SetFit/Transformers import breakage.
!pip install -q -U "setfit>=1.1.1" "transformers>=4.46,<5" "accelerate>=0.30" datasets

# ── 3. Standard imports ───────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
import torch
import setfit
import transformers
from setfit import SetFitModel, Trainer, TrainingArguments

# ── 2. Mount Google Drive (optional) ─────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/cs4120_project/"
except Exception:
    SAVE_DIR = None

# ── 4. Sanity checks ──────────────────────────────────────────────
print(f"PyTorch: {torch.__version__}")
print(f"SetFit: {setfit.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 41.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Mounted at /content/drive
PyTorch: 2.10.0+cu128
GPU available: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Shared Evaluation Wiring

This section integrates `src/evaluate.py` so SetFit uses the same metrics/report format as other models.
Run this cell once, then call `evaluate_setfit_predictions(...)` after generating `y_pred_binary` for test data.


In [ ]:
import ast
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.preprocessing import MultiLabelBinarizer

# Resolve repo root so src/ imports work in both local Jupyter and Colab
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src" / "evaluate.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find repo root containing src/evaluate.py")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluate import evaluate_run, save_evaluation_outputs

def _parse_labels_column(df, label_col="labels"):
    df = df.copy()

    def _parse_one(value):
        if isinstance(value, list):
            return [int(x) for x in value]
        if isinstance(value, np.ndarray):
            return [int(x) for x in value.tolist()]
        if isinstance(value, str):
            s = value.strip()
            if s.startswith('[') and s.endswith(']'):
                body = s[1:-1].strip()
                if body == "":
                    return []
                if ',' in body:
                    tokens = [t.strip() for t in body.split(',') if t.strip()]
                else:
                    tokens = [t for t in body.split() if t]
                return [int(t) for t in tokens]
            return [int(x) for x in ast.literal_eval(s)]
        return value

    df[label_col] = df[label_col].apply(_parse_one)
    return df

def _resolve_data_dir():
    # Local notebooks usually run from notebooks/, so data is often ../data
    for candidate in [repo_root / "data", Path("../data"), Path("data")]:
        if (candidate / "test.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data directory containing test.csv")

def _get_label_names():
    dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
    return dataset["train"].features["labels"].feature.names

def evaluate_setfit_predictions(
    y_pred_binary,
    *,
    data_fraction,
    seed,
    method="setfit",
    data_dir=None,
    output_dir=None,
):
    """
    Evaluate SetFit predictions on the shared test split and save standard outputs.

    y_pred_binary: binary matrix (n_samples, n_labels) aligned with test.csv rows.
    """
    data_dir = Path(data_dir) if data_dir else _resolve_data_dir()
    output_dir = Path(output_dir) if output_dir else (repo_root / "results")

    test_df = pd.read_csv(data_dir / "test.csv")
    test_df = _parse_labels_column(test_df, label_col="labels")

    label_names = _get_label_names()
    mlb = MultiLabelBinarizer(classes=list(range(len(label_names))))
    mlb.fit([list(range(len(label_names)))])
    y_true_binary = mlb.transform(test_df["labels"])

    evaluation = evaluate_run(
        method=method,
        data_fraction=float(data_fraction),
        seed=int(seed),
        label_names=label_names,
        y_true=y_true_binary,
        y_pred=y_pred_binary,
    )

    paths = save_evaluation_outputs(evaluation, method=method, output_dir=output_dir)
    return evaluation, paths

print("Shared evaluation helpers loaded.")
print("After generating SetFit predictions, call evaluate_setfit_predictions(y_pred_binary, data_fraction=..., seed=...)")


## 3. SetFit Experiment Config

Configure fractions, seeds, model, and output paths for SetFit runs.


In [ ]:
# --- SetFit experiment config ---
METHOD_NAME = "setfit"
MODEL_NAME = "sentence-transformers/paraphrase-mpnet-base-v2"

# Include full-data run (1.00) for low-data vs full-data comparisons
DATA_FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS = [42, 7, 21]

TRAIN_FILE_BY_FRACTION = {
    0.01: "train_1pct.csv",
    0.05: "train_5pct.csv",
    0.10: "train_10pct.csv",
    0.25: "train_25pct.csv",
    0.50: "train_50pct.csv",
    1.00: "train.csv",
}

SETFIT_CONFIG = {
    "num_iterations": 20,
    "num_epochs": 1,
    "batch_size": 16,
    "body_learning_rate": 2e-5,
    "head_learning_rate": 1e-2,
}

RESULTS_DIR = repo_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Optional checkpoint save for full-data runs (if SAVE_DIR exists from setup cell)
ENABLE_CHECKPOINT_SAVE = ("SAVE_DIR" in globals()) and bool(SAVE_DIR)
if ENABLE_CHECKPOINT_SAVE:
    CHECKPOINT_ROOT = Path(SAVE_DIR) / "checkpoints" / "setfit"
    CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

print("Config loaded.")
print("Results dir:", RESULTS_DIR)


## 4. Run SetFit Experiments

Trains SetFit across configured fractions and seeds, evaluates with shared framework, and writes result CSVs.


In [ ]:
import gc
import os
import random
from datasets import Dataset

# Disable W&B prompts in notebook runs
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

def _set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

def _choose_text_col(df):
    for col in ["text_clean_transformer", "text"]:
        if col in df.columns:
            return col
    raise ValueError("No text column found. Expected one of: text_clean_transformer, text")

def _to_binary_matrix(label_lists, n_labels):
    mlb = MultiLabelBinarizer(classes=list(range(n_labels)))
    mlb.fit([list(range(n_labels))])
    return mlb.transform(label_lists)

def _predictions_to_binary(preds, n_labels):
    # Case 1: already matrix-like
    try:
        arr = np.asarray(preds)
        if arr.ndim == 2 and arr.shape[1] == n_labels:
            return (arr > 0).astype(int)
    except Exception:
        pass

    # Case 2: sequence of predicted label-id collections
    out = np.zeros((len(preds), n_labels), dtype=int)
    for i, pred in enumerate(preds):
        if isinstance(pred, (list, tuple, set, np.ndarray)):
            for label_id in pred:
                try:
                    idx = int(label_id)
                    if 0 <= idx < n_labels:
                        out[i, idx] = 1
                except Exception:
                    continue
        else:
            try:
                idx = int(pred)
                if 0 <= idx < n_labels:
                    out[i, idx] = 1
            except Exception:
                continue
    return out

def _build_setfit_model(model_name):
    # Prefer explicit multilabel strategy when supported
    try:
        return SetFitModel.from_pretrained(model_name, multi_target_strategy="one-vs-rest")
    except TypeError:
        return SetFitModel.from_pretrained(model_name)

def _build_trainer(model, train_dataset, eval_dataset, cfg):
    last_exc = None

    # Newer API path
    try:
        args = TrainingArguments(
            batch_size=cfg["batch_size"],
            num_epochs=cfg["num_epochs"],
            num_iterations=cfg["num_iterations"],
            body_learning_rate=cfg["body_learning_rate"],
            head_learning_rate=cfg["head_learning_rate"],
            report_to="none",
        )
        return Trainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            column_mapping={"text": "text", "label": "label"},
        )
    except Exception as exc:
        last_exc = exc

    # Legacy API fallback
    try:
        return Trainer(
            model=model,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            batch_size=cfg["batch_size"],
            num_epochs=cfg["num_epochs"],
            num_iterations=cfg["num_iterations"],
            column_mapping={"text": "text", "label": "label"},
        )
    except Exception as exc:
        last_exc = exc

    raise RuntimeError(f"Could not construct SetFit Trainer with known APIs: {last_exc}")

def run_single_setfit_experiment(data_fraction, seed):
    data_dir = _resolve_data_dir()

    train_filename = TRAIN_FILE_BY_FRACTION[float(data_fraction)]
    train_path = data_dir / train_filename
    test_path = data_dir / "test.csv"

    if not train_path.exists():
        raise FileNotFoundError(f"Missing train file for fraction={data_fraction}: {train_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Missing test file: {test_path}")

    train_df = _parse_labels_column(pd.read_csv(train_path), label_col="labels")
    test_df = _parse_labels_column(pd.read_csv(test_path), label_col="labels")

    text_col = _choose_text_col(train_df)
    label_names = _get_label_names()
    n_labels = len(label_names)

    y_train = _to_binary_matrix(train_df["labels"], n_labels)
    y_test = _to_binary_matrix(test_df["labels"], n_labels)

    train_dataset = Dataset.from_dict({
        "text": train_df[text_col].fillna("").astype(str).tolist(),
        "label": y_train.tolist(),
    })
    eval_dataset = Dataset.from_dict({
        "text": test_df[text_col].fillna("").astype(str).tolist(),
        "label": y_test.tolist(),
    })

    _set_global_seed(int(seed))
    model = _build_setfit_model(MODEL_NAME)
    trainer = _build_trainer(model, train_dataset, eval_dataset, SETFIT_CONFIG)
    trainer.train()

    raw_preds = model.predict(test_df[text_col].fillna("").astype(str).tolist())
    y_pred = _predictions_to_binary(raw_preds, n_labels)

    evaluation = evaluate_run(
        method=METHOD_NAME,
        data_fraction=float(data_fraction),
        seed=int(seed),
        label_names=label_names,
        y_true=y_test,
        y_pred=y_pred,
    )

    if ENABLE_CHECKPOINT_SAVE and float(data_fraction) == 1.00 and SAVE_DIR:
        ckpt_dir = CHECKPOINT_ROOT / f"seed_{seed}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(ckpt_dir))

    # Best-effort memory cleanup before returning
    try:
        del trainer
        del model
    except Exception:
        pass
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

    return evaluation


## 5. Resumable Runner (Crash-Safe)

Loads existing result files, tracks completed runs, and persists after every successful run.


In [ ]:
OVERALL_PATH = RESULTS_DIR / "setfit_overall.csv"
PER_CLASS_PATH = RESULTS_DIR / "setfit_per_class.csv"
README_PATH = RESULTS_DIR / "setfit_results.csv"
ERRORS_PATH = RESULTS_DIR / "setfit_errors.csv"

def _safe_read_csv(path):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()

overall_df = _safe_read_csv(OVERALL_PATH)
per_class_df = _safe_read_csv(PER_CLASS_PATH)
errors_df = _safe_read_csv(ERRORS_PATH)

def _dedup_overall(df):
    if df.empty:
        return df
    return (
        df.sort_values(["method", "seed", "data_fraction"])
          .drop_duplicates(subset=["method", "seed", "data_fraction"], keep="last")
          .reset_index(drop=True)
    )

def _dedup_per_class(df):
    if df.empty:
        return df
    return (
        df.sort_values(["method", "seed", "data_fraction", "emotion"])
          .drop_duplicates(subset=["method", "seed", "data_fraction", "emotion"], keep="last")
          .reset_index(drop=True)
    )

def _persist_results():
    global overall_df, per_class_df, errors_df

    overall_df = _dedup_overall(overall_df)
    per_class_df = _dedup_per_class(per_class_df)

    overall_df.to_csv(OVERALL_PATH, index=False)
    per_class_df.to_csv(PER_CLASS_PATH, index=False)

    if not per_class_df.empty:
        readme_df = per_class_df[["method", "data_fraction", "seed", "emotion", "f1", "precision", "recall"]].copy()
        readme_df.to_csv(README_PATH, index=False)

    if not errors_df.empty:
        errors_df.to_csv(ERRORS_PATH, index=False)

def _completed_pairs():
    if overall_df.empty:
        return set()
    return set((float(r.data_fraction), int(r.seed)) for r in overall_df[["data_fraction", "seed"]].itertuples(index=False))

def show_setfit_progress():
    done = _completed_pairs()
    expected = [(float(f), int(s)) for s in SEEDS for f in DATA_FRACTIONS]
    pending = [p for p in expected if p not in done]

    print(f"Completed runs: {len(done)} / {len(expected)}")
    if pending:
        print("Pending:", pending)
    else:
        print("No pending runs.")

    if not overall_df.empty:
        display(overall_df.sort_values(["seed", "data_fraction"]).tail(10))

show_setfit_progress()


## 6. Run One Job or Small Batch

Use these helpers to run one `(fraction, seed)` at a time or a bounded number of pending runs.


In [ ]:
def run_and_persist_one(data_fraction, seed, *, force=False):
    global overall_df, per_class_df, errors_df

    key = (float(data_fraction), int(seed))
    if (not force) and key in _completed_pairs():
        print(f"Skipping completed run: {key}")
        return

    print(f"Running SetFit: fraction={data_fraction}, seed={seed}")
    try:
        eval_out = run_single_setfit_experiment(data_fraction, seed)

        # Remove old rows for this key before append (idempotent reruns)
        if not overall_df.empty:
            overall_df = overall_df[~((overall_df["data_fraction"].astype(float) == float(data_fraction)) & (overall_df["seed"].astype(int) == int(seed)))]
        if not per_class_df.empty:
            per_class_df = per_class_df[~((per_class_df["data_fraction"].astype(float) == float(data_fraction)) & (per_class_df["seed"].astype(int) == int(seed)))]

        overall_df = pd.concat([overall_df, eval_out["overall"]], ignore_index=True) if not overall_df.empty else eval_out["overall"].copy()
        per_class_df = pd.concat([per_class_df, eval_out["per_class"]], ignore_index=True) if not per_class_df.empty else eval_out["per_class"].copy()

        _persist_results()

        macro_f1 = float(eval_out["overall"].iloc[0]["macro_f1"])
        print(f"Completed and saved: macro_f1={macro_f1:.4f}")

    except Exception as exc:
        err_row = pd.DataFrame([{"seed": int(seed), "data_fraction": float(data_fraction), "error": str(exc)}])
        errors_df = pd.concat([errors_df, err_row], ignore_index=True) if not errors_df.empty else err_row
        _persist_results()
        print(f"FAILED: {exc}")

def run_pending(*, fractions=None, seeds=None, max_runs=None):
    fracs = [float(f) for f in (fractions if fractions is not None else DATA_FRACTIONS)]
    seed_list = [int(s) for s in (seeds if seeds is not None else SEEDS)]

    planned = [(f, s) for s in seed_list for f in fracs]
    pending = [p for p in planned if p not in _completed_pairs()]

    if max_runs is not None:
        pending = pending[: int(max_runs)]

    print(f"Pending selected runs: {len(pending)}")
    for f, s in pending:
        run_and_persist_one(f, s)

    show_setfit_progress()


## 7. Execution Examples

Uncomment one block at a time so runs are split across cells/sessions.


In [ ]:
# Example A: run a single job
# run_and_persist_one(0.50, 42)

# Example B: run one fraction across all seeds
# run_pending(fractions=[0.25])

# Example C: run next 2 pending jobs only (useful with limited RAM/credits)
# run_pending(max_runs=2)

# Example D: continue all pending jobs
# run_pending()

show_setfit_progress()
